# Hindi SLM Training Walkthrough

**Architecture:** ~68M parameter decoder-only transformer (GQA + RoPE + RMSNorm + SwiGLU)  
**Target:** Raspberry Pi 8 GB (Q4_K_M GGUF)  
**Data:** ai4bharat/sangraha (hin/verified) — 1 GB downloaded once  
**Tokenizer:** `vaibhavmaurya/hindi-slm-tokenizer-v001` (32k Unigram, frozen)  

## Stages
| # | Stage | Status |
|---|-------|--------|
| 0 | Environment Setup | |
| 1 | System Profiling | |
| 2 | Architecture Deduction | |
| 3 | Data Download (one-time) | |
| 4 | Tokenizer Freezing | |
| 5 | Clean + Tokenize + Pack | |
| 6 | Model Architecture | |
| 7 | Training | |
| 8 | Evaluation | |
| 9 | Edge Export | |

---
## Section 0 — Environment Setup

In [ ]:
# Install dependencies (run once)
# If you have a CUDA-enabled llama-cpp-python build, install it separately:
#   CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "transformers>=4.40", "datasets>=2.18", "accelerate>=0.28",
    "huggingface_hub", "tensorboard", "rich", "pyyaml", "pyarrow", "tqdm",
], check=True)
print("Dependencies installed.")

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [2]:
import sys, os
from pathlib import Path

# Resolve project root (notebook lives in slm_training/notebooks/)
NOTEBOOK_DIR = Path().resolve()
SLM_TRAINING_ROOT = NOTEBOOK_DIR.parent
PROJECT_ROOT = SLM_TRAINING_ROOT.parent

# Add src to path so slm_training package is importable
SRC_DIR = SLM_TRAINING_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Key paths
CONFIGS_DIR      = SLM_TRAINING_ROOT / "configs"
DATA_RAW_DIR     = SLM_TRAINING_ROOT / "data" / "raw"
DATA_TOK_DIR     = SLM_TRAINING_ROOT / "data" / "tokenized"
ARTIFACTS_DIR    = SLM_TRAINING_ROOT / "artifacts"
CKPT_DIR         = ARTIFACTS_DIR / "checkpoints"
MODELS_DIR       = ARTIFACTS_DIR / "models"
REPORTS_DIR      = ARTIFACTS_DIR / "reports"
TB_DIR           = SLM_TRAINING_ROOT / "logs" / "tensorboard"

# Create directories
for d in [CONFIGS_DIR, DATA_RAW_DIR, DATA_TOK_DIR, ARTIFACTS_DIR, CKPT_DIR, MODELS_DIR, REPORTS_DIR, TB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"SLM_TRAINING_ROOT : {SLM_TRAINING_ROOT}")
print(f"PROJECT_ROOT      : {PROJECT_ROOT}")

SLM_TRAINING_ROOT : C:\Users\vaibh\Documents\LLMs\projects\SLM_HINDI\slm_training
PROJECT_ROOT      : C:\Users\vaibh\Documents\LLMs\projects\SLM_HINDI


In [3]:
# =============================================================================
# RUN MODE — set this before executing any other section
# "TEST" : downloads ~10 MB, trains 100 steps — validates the full pipeline in
#          ~5-10 minutes. Loss will not converge; goal is end-to-end correctness.
# "FULL" : downloads 1 GB, trains 50,000 steps — real production run (~3-7 days).
# =============================================================================
RUN_MODE = "FULL"   # <-- change to "FULL" for production training

if RUN_MODE == "TEST":
    print("=" * 62)
    print("  TEST MODE — end-to-end pipeline smoke test")
    print("  Data   : ~10 MB sample from ai4bharat/sangraha")
    print("  Steps  : 100  |  no gradient accumulation")
    print("  Goal   : verify every stage runs without errors")
    print("  ETA    : ~5-10 minutes on GPU")
    print("=" * 62)
    _TARGET_BYTES  = 10 * 1024 * 1024   # 10 MB
    _DATASET_NAME  = "sangraha_hin_test"
    _MAX_STEPS     = 100
    _WARMUP_STEPS  = 10
    _GRAD_ACCUM    = 1
    _BATCH_SIZE    = 2
    _LOG_EVERY     = 5
    _EVAL_EVERY    = 25
    _SAVE_EVERY    = 50
    _NUM_PROC      = 2
    _NUM_WORKERS   = 2
    # Separate artifact paths so test and full runs never overwrite each other
    DATA_TOK_DIR   = SLM_TRAINING_ROOT / "data" / "tokenized_test"
    CKPT_DIR       = ARTIFACTS_DIR / "checkpoints_test"
    MANIFEST_PATH  = CONFIGS_DIR / "tokenized_dataset_manifest_test.json"

elif RUN_MODE == "FULL":
    print("=" * 62)
    print("  FULL MODE — production training")
    print("  Data   : 1 GB from ai4bharat/sangraha")
    print("  Steps  : 50,000  |  effective batch = 2 × 16 = 32")
    print("  ETA    : ~3-7 days on laptop GPU")
    print("=" * 62)
    _TARGET_BYTES  = 10 * 1024 ** 3      # 1 GB
    _DATASET_NAME  = "sangraha_hin_10gb"
    _MAX_STEPS     = 50_000
    _WARMUP_STEPS  = 1_000
    _GRAD_ACCUM    = 16
    _BATCH_SIZE    = 2
    _LOG_EVERY     = 50
    _EVAL_EVERY    = 500
    _SAVE_EVERY    = 1_000
    _NUM_PROC      = 6
    _NUM_WORKERS   = 6
    DATA_TOK_DIR   = SLM_TRAINING_ROOT / "data" / "tokenized"
    CKPT_DIR       = ARTIFACTS_DIR / "checkpoints"
    MANIFEST_PATH  = CONFIGS_DIR / "tokenized_dataset_manifest.json"

else:
    raise ValueError(f"RUN_MODE must be 'TEST' or 'FULL', got '{RUN_MODE}'")

# Create mode-specific directories
for d in [DATA_TOK_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"\nDATA_TOK_DIR : {DATA_TOK_DIR}")
print(f"CKPT_DIR     : {CKPT_DIR}")
print(f"MANIFEST     : {MANIFEST_PATH}")

  FULL MODE — production training
  Data   : 1 GB from ai4bharat/sangraha
  Steps  : 50,000  |  effective batch = 2 × 16 = 32
  ETA    : ~3-7 days on laptop GPU

DATA_TOK_DIR : C:\Users\vaibh\Documents\LLMs\projects\SLM_HINDI\slm_training\data\tokenized
CKPT_DIR     : C:\Users\vaibh\Documents\LLMs\projects\SLM_HINDI\slm_training\artifacts\checkpoints
MANIFEST     : C:\Users\vaibh\Documents\LLMs\projects\SLM_HINDI\slm_training\configs\tokenized_dataset_manifest.json


In [4]:
import torch
import os

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM             : {props.total_memory / 1e9:.2f} GB")
    print(f"CUDA version     : {torch.version.cuda}")
    print(f"bfloat16 support : {torch.cuda.is_bf16_supported()}")
    DEVICE = torch.device("cuda")
    DTYPE  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    print("WARNING: No GPU detected. Training will run on CPU (very slow).")
    DEVICE = torch.device("cpu")
    DTYPE  = torch.float32

print(f"\nUsing device={DEVICE}, dtype={DTYPE}")

PyTorch version  : 2.6.0+cu124
CUDA available   : True
GPU              : NVIDIA RTX 3000 Ada Generation Laptop GPU
VRAM             : 8.59 GB
CUDA version     : 12.4
bfloat16 support : True

Using device=cuda, dtype=torch.bfloat16


---
## Section 1 — System Profiling

In [5]:
from slm_training.profiler import detect_profile, save_profile, print_profile_table

profile = detect_profile(disk_path=str(SLM_TRAINING_ROOT))
save_profile(profile, CONFIGS_DIR / "system_profile.yaml")
print_profile_table(profile)

TRAINING_TIER = profile.training_tier
print(f"\nTraining tier: {TRAINING_TIER}")

                           System Profile                           
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Component            ┃ Value                                     ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ OS                   │ Windows 11                                │
│ Python               │ 3.12.5                                    │
│ CPU                  │ 13th Gen Intel(R) Core(TM) i7-13700H      │
│ CPU Cores (physical) │ 14                                        │
│ CPU Threads          │ 20                                        │
│ RAM                  │ 34.0 GB                                   │
│ GPU Available        │ Yes                                       │
│ GPU Name             │ NVIDIA RTX 3000 Ada Generation Laptop GPU │
│ GPU VRAM             │ 8.59 GB                                   │
│ GPU Architecture     │ Ada Lovelace                              │
│ CUDA Version         │ 12.4                                      │
│ bfloat16 Supported   │ Yes                                       │
│ Free Disk            │ 402.0 GB                                  │
│ Training Tier        │ SMALL                                     │
└──────────────────────┴───────────────────────────────────────────┘


Training tier: SMALL


---
## Section 2 — Architecture Deduction

In [6]:
from slm_training.architecture import (
    NASCalculator, ParameterCounter, MemoryEstimator, KVCacheEstimator,
    save_model_schema, TIER_CONFIGS
)

# Select architecture for detected tier
# Override to "SMALL" to always use the Pi-target 68M config regardless of detected tier
TARGET_TIER = "SMALL"  # change to TRAINING_TIER to auto-select
MODEL_CFG = NASCalculator.select(TARGET_TIER)

NASCalculator.print_summary(MODEL_CFG)

param_counts = ParameterCounter.count(MODEL_CFG)
print(f"\nTotal parameters: {param_counts['total'] / 1e6:.2f}M")
# assert 60_000_000 < param_counts['total'] < 120_000_000, (
#     f"Unexpected param count: {param_counts['total']/1e6:.2f}M. Check ModelConfig."
# )

save_model_schema(MODEL_CFG, param_counts, CONFIGS_DIR / "optimal_model_schema.yaml")
print("Model schema written to configs/optimal_model_schema.yaml")

       Model Architecture        
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Parameter           ┃ Value   ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ hidden_size         │ 512     │
│ num_layers          │ 10      │
│ num_attention_heads │ 8       │
│ num_kv_heads        │ 2       │
│ head_dim            │ 64      │
│ intermediate_size   │ 1536    │
│ max_seq_len         │ 512     │
│ vocab_size          │ 32000   │
│ rms_norm_eps        │ 1e-06   │
│ rope_base           │ 10000.0 │
│ tie_embeddings      │ True    │
│ dropout             │ 0.1     │
└─────────────────────┴─────────┘

               Parameter Budget                
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ Component           ┃ Count      ┃ M params ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━┩
│ embedding           │ 16,384,000 │ 16.38    │
│ attention_per_layer │ 655,360    │ 0.66     │
│ mlp_per_layer       │ 2,359,296  │ 2.36     │
│ norm_per_layer      │ 1,024      │ 0.00     │
│ all_layers          │ 30,156,800 │ 30.16    │
│ final_norm          │ 512        │ 0.00     │
│ lm_head             │ 0          │ 0.00     │
│ total               │ 46,541,312 │ 46.54    │
└─────────────────────┴────────────┴──────────┘

VRAM Estimate (batch=2, bfloat16)
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Component            ┃ GB     ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ weights_gb           │ 0.093  │
│ gradients_gb         │ 0.093  │
│ adam_states_gb       │ 0.372  │
│ activations_gb       │ 0.252  │
│ total_estimated_gb   │ 0.81   │
│ KV cache (inference) │ 0.0026 │
└──────────────────────┴────────┘


Total parameters: 46.54M
Model schema written to configs/optimal_model_schema.yaml


---
## Section 3 — Load Local Sangraha Data

Loads pre-downloaded Hindi text from local `.jsonl.gz` files.  
Data is in `data/sangraha_verified_hin_10gb/` (downloaded separately via `download_sangraha_10gb.py`).

| RUN_MODE | Files loaded | Approx rows |
|----------|-------------|-------------|
| `TEST`   | 1 file      | ~95,000     |
| `FULL`   | All 5 files | ~475,000    |

In [7]:
# from slm_training.dataset import download_sangraha
# import os


# from datasets import load_dataset

# dataset = load_dataset("ai4bharat/sangraha", data_dir="<subset_name>/<lang_code>")


# RAW_DATASET_PATH = download_sangraha(
#     raw_dir=DATA_RAW_DIR,
#     data_plan_path=CONFIGS_DIR / "data_plan.yaml",
#     target_bytes = 10 * 1024 ** 3 ,
#     dataset_name = "sangraha_hin_10gb",    
# )
# print(f"Raw dataset path: {RAW_DATASET_PATH}")

In [ ]:
from slm_training.dataset import load_local_sangraha

LOCAL_SANGRAHA_DIR = SLM_TRAINING_ROOT / "data" / "sangraha_verified_hin_10gb"

# TEST: load 1 file (~95k rows); FULL: load all 5 files (~475k rows)
_MAX_FILES = 1 if RUN_MODE == "TEST" else None

print(f"_MAX_FILES = {_MAX_FILES}")

raw_ds = load_local_sangraha(LOCAL_SANGRAHA_DIR, max_files=_MAX_FILES)
print(f"Rows loaded : {len(raw_ds):,}")
print(f"Columns     : {raw_ds.column_names}")
for i in range(min(2, len(raw_ds))):
    txt = raw_ds[i]["text"]
    print(f"  [{i}] {txt[:120]}...")
assert len(raw_ds) > 0, "No rows loaded!"
print("PASS: local data loaded")

_MAX_FILES = None


In [ ]:
import yaml
from transformers import AutoTokenizer

# Try local path first (from tokenizer_training workstream)
LOCAL_TOK_PATH = PROJECT_ROOT / "tokenizer_training" / "data" / "final" / "hindi_slm_tokenizer_v001"
HF_TOK_NAME    = "vaibhavmaurya/hindi-slm-tokenizer-v001"

if LOCAL_TOK_PATH.exists():
    print(f"Loading tokenizer from local path: {LOCAL_TOK_PATH}")
    tokenizer = AutoTokenizer.from_pretrained(str(LOCAL_TOK_PATH))
else:
    print(f"Loading tokenizer from HuggingFace Hub: {HF_TOK_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(HF_TOK_NAME)

print(f"Vocab size       : {tokenizer.vocab_size}")
print(f"Model max length : {tokenizer.model_max_length}")

# Assertions
assert tokenizer.vocab_size == 32000, f"Expected 32000, got {tokenizer.vocab_size}"

required_special = ["<pad>", "<unk>", "<s>", "</s>"]
for tok in required_special:
    assert tok in tokenizer.get_vocab(), f"Missing special token: {tok}"
print(f"Special tokens   : {tokenizer.all_special_tokens}")
print("Tokenizer assertions PASSED")

In [ ]:
# Round-trip test
test_sentences = [
    "नमस्ते! भारत एक महान देश है।",
    "हिंदी भाषा बहुत सुंदर है और इसमें लाखों शब्द हैं।",
    "आज मौसम बहुत अच्छा है। बाहर धूप निकली हुई है।",
    "विज्ञान और तकनीक के क्षेत्र में भारत ने बहुत प्रगति की है।",
    "रामायण और महाभारत भारतीय संस्कृति के महान ग्रंथ हैं।",
]

print("Round-trip tests:")
all_pass = True
for sent in test_sentences:
    ids = tokenizer.encode(sent, add_special_tokens=False)
    decoded = tokenizer.decode(ids, skip_special_tokens=True)
    unk_count = ids.count(tokenizer.unk_token_id)
    ok = decoded.strip() == sent.strip() and unk_count == 0
    status = "PASS" if ok else "FAIL"
    if not ok:
        all_pass = False
    print(f"  [{status}] tokens={len(ids):3d}  unk={unk_count}  text='{sent[:50]}'")

print(f"\nAll round-trips: {'PASS' if all_pass else 'FAIL (check tokenizer)'}")

# Sync vocab size into model config
MODEL_CFG.vocab_size = tokenizer.vocab_size

# Write tokenizer config
import hashlib, json
tok_config = {
    "tokenizer_name": HF_TOK_NAME,
    "vocab_size": tokenizer.vocab_size,
    "algorithm": "Unigram",
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token_id": tokenizer.pad_token_id,
    "bos_token_id": tokenizer.bos_token_id,
}
with open(CONFIGS_DIR / "tokenizer_config.yaml", "w", encoding="utf-8") as f:
    yaml.dump(tok_config, f, default_flow_style=False, allow_unicode=True)
print("tokenizer_config.yaml written.")

from slm_training.dataset import build_tokenized_splits

# Check if already tokenized for this RUN_MODE (skip if manifest exists)
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
    print(f"[{RUN_MODE}] Tokenized splits already exist — skipping tokenization.")
    print(f"  train={manifest['splits']['train']:,}  val={manifest['splits']['val']:,}  test={manifest['splits']['test']:,}")
else:
    manifest = build_tokenized_splits(
        raw_dataset_path=RAW_DATASET_PATH,
        tokenized_dir=DATA_TOK_DIR,
        manifest_path=MANIFEST_PATH,
        tokenizer=tokenizer,
        seq_len=MODEL_CFG.max_seq_len,
        seed=42,
        num_proc=_NUM_PROC,
    )

print(f"\nManifest:")
for k, v in manifest.items():
    print(f"  {k}: {v}")

In [ ]:
from slm_training.dataset import build_tokenized_splits

# Check if already tokenized (skip if manifest exists)
MANIFEST_PATH = CONFIGS_DIR / "tokenized_dataset_manifest.json"

if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
    print("Tokenized splits already exist. Skipping tokenization.")
    print(f"  train={manifest['splits']['train']:,}  val={manifest['splits']['val']:,}  test={manifest['splits']['test']:,}")
else:
    manifest = build_tokenized_splits(
        raw_dataset_path=RAW_DATASET_PATH,
        tokenized_dir=DATA_TOK_DIR,
        manifest_path=MANIFEST_PATH,
        tokenizer=tokenizer,
        seq_len=MODEL_CFG.max_seq_len,
        seed=42,
        num_proc=6,
    )

print(f"\nManifest:")
for k, v in manifest.items():
    print(f"  {k}: {v}")

---
## Section 6 — Model Architecture

In [ ]:
from slm_training.dataset import build_tokenized_splits

if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
    print(f"[{RUN_MODE}] Tokenized splits already exist — skipping.")
    print(f"  train={manifest['splits']['train']:,}  val={manifest['splits']['val']:,}  test={manifest['splits']['test']:,}")
else:
    manifest = build_tokenized_splits(
        raw_dataset_or_path=raw_ds,
        tokenized_dir=DATA_TOK_DIR,
        manifest_path=MANIFEST_PATH,
        tokenizer=tokenizer,
        seq_len=MODEL_CFG.max_seq_len,
        seed=42,
        num_proc=_NUM_PROC,
    )

print(f"\nManifest:")
for k, v in manifest.items():
    print(f"  {k}: {v}")

In [ ]:
# Instantiate model
model = HindiSLM(MODEL_CFG)

total_params = model.count_parameters()
print(f"Total parameters : {total_params:,} ({total_params/1e6:.2f}M)")

# Breakdown by module
for name, module in [("embed_tokens", model.embed_tokens), ("layers[0]", model.layers[0]), ("norm", model.norm)]:
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:20s}: {n:,} params")

In [ ]:
---
## Section 7 — Training

Training parameters are controlled by `RUN_MODE` set in Section 0.

| Setting | TEST | FULL |
|---------|------|------|
| Max steps | 100 | 50,000 |
| Gradient accumulation | 1 (eff. batch=2) | 16 (eff. batch=32) |
| Eval every | 25 steps | 500 steps |
| Save every | 50 steps | 1,000 steps |
| Checkpoint dir | `checkpoints_test/` | `checkpoints/` |

- Mixed precision: `bfloat16`
- OOM recovery: auto-adjusts batch, enables gradient checkpointing, reduces seq_len
- Checkpoints auto-resume — safe to interrupt and re-run at any time
- TensorBoard logs to `logs/tensorboard/`

from slm_training.trainer import TrainingConfig, save_training_config

TRAIN_CFG = TrainingConfig(
    learning_rate=3e-4,
    weight_decay=0.1,
    max_steps=_MAX_STEPS,
    warmup_steps=_WARMUP_STEPS,
    per_device_batch_size=_BATCH_SIZE,
    gradient_accumulation_steps=_GRAD_ACCUM,
    max_grad_norm=1.0,
    num_workers=_NUM_WORKERS,
    log_every=_LOG_EVERY,
    eval_every=_EVAL_EVERY,
    save_every=_SAVE_EVERY,
    dtype="bfloat16",
    max_oom_retries=5,
)

cfg_name = f"training_config_{RUN_MODE.lower()}.yaml"
save_training_config(TRAIN_CFG, CONFIGS_DIR / cfg_name)
print(f"[{RUN_MODE}] Max steps          : {TRAIN_CFG.max_steps:,}")
print(f"[{RUN_MODE}] Effective batch    : {TRAIN_CFG.per_device_batch_size} × {TRAIN_CFG.gradient_accumulation_steps} = {TRAIN_CFG.per_device_batch_size * TRAIN_CFG.gradient_accumulation_steps}")
print(f"[{RUN_MODE}] Warmup steps       : {TRAIN_CFG.warmup_steps}")
print(f"{cfg_name} written.")

In [ ]:
# Build DataLoaders
from slm_training.dataset import make_dataloader

train_loader = make_dataloader(
    DATA_TOK_DIR / "train",
    batch_size=TRAIN_CFG.per_device_batch_size,
    num_workers=_NUM_WORKERS,
    shuffle=True,
)
val_loader = make_dataloader(
    DATA_TOK_DIR / "val",
    batch_size=TRAIN_CFG.per_device_batch_size,
    num_workers=min(2, _NUM_WORKERS),
    shuffle=False,
)

print(f"[{RUN_MODE}] Train batches : {len(train_loader):,}")
print(f"[{RUN_MODE}] Val batches   : {len(val_loader):,}")

# Verify a batch
sample_batch = next(iter(train_loader))
print(f"Batch shape   : {sample_batch['input_ids'].shape}")

In [ ]:
# Build DataLoaders
from slm_training.dataset import make_dataloader

train_loader = make_dataloader(
    DATA_TOK_DIR / "train",
    batch_size=TRAIN_CFG.per_device_batch_size,
    num_workers=TRAIN_CFG.num_workers,
    shuffle=True,
)
val_loader = make_dataloader(
    DATA_TOK_DIR / "val",
    batch_size=TRAIN_CFG.per_device_batch_size,
    num_workers=2,
    shuffle=False,
)

print(f"Train batches : {len(train_loader):,}")
print(f"Val batches   : {len(val_loader):,}")

# Verify a batch
sample_batch = next(iter(train_loader))
print(f"Batch shape   : {sample_batch['input_ids'].shape}")

In [ ]:
# Setup TensorBoard
try:
    from torch.utils.tensorboard import SummaryWriter
    tb_writer = SummaryWriter(log_dir=str(TB_DIR))
    print(f"TensorBoard writer created. Run:\n  tensorboard --logdir {TB_DIR}")
except Exception as e:
    tb_writer = None
    print(f"TensorBoard not available: {e}")

In [ ]:
# ===== START TRAINING =====
# This cell runs the full training loop.
# Safe to interrupt (Ctrl+C) — will save checkpoint and resume on re-run.
# Monitor progress via TensorBoard or the rich progress bar.

from slm_training.trainer import train

train(
    model=model,
    model_cfg=MODEL_CFG,
    train_cfg=TRAIN_CFG,
    train_loader=train_loader,
    val_loader=val_loader,
    ckpt_dir=CKPT_DIR,
    device=DEVICE,
    tb_writer=tb_writer,
)

---
## Section 8 — Evaluation

In [ ]:
import json
from slm_training.trainer import evaluate_loss, _find_latest_checkpoint, _load_checkpoint, _build_optimizer
from slm_training.evaluator import compute_perplexity, generate_samples, print_samples, write_evaluation_report

# Load best/latest checkpoint
latest_ckpt = _find_latest_checkpoint(CKPT_DIR)
if latest_ckpt:
    optimizer = _build_optimizer(model, TRAIN_CFG)
    step = _load_checkpoint(latest_ckpt, model, optimizer)
    model = model.to(DEVICE)
    print(f"Loaded checkpoint: {latest_ckpt.name} (step {step})")
else:
    step = 0
    print("No checkpoint found — evaluating untrained model.")

In [ ]:
# Validation perplexity
val_ppl = compute_perplexity(model, val_loader, DEVICE, DTYPE, max_batches=200)
print(f"Validation perplexity: {val_ppl:.2f}")

# Test perplexity
test_loader = make_dataloader(DATA_TOK_DIR / "test", batch_size=TRAIN_CFG.per_device_batch_size, num_workers=2, shuffle=False)
test_ppl = compute_perplexity(model, test_loader, DEVICE, DTYPE, max_batches=100)
print(f"Test perplexity     : {test_ppl:.2f}")

In [ ]:
# Hindi generation samples
samples = generate_samples(
    model, tokenizer, DEVICE,
    max_new_tokens=80,
    temperature=0.8,
    top_p=0.9,
)
print_samples(samples)

In [ ]:
# Write evaluation report
write_evaluation_report(
    report_path=REPORTS_DIR / "evaluation_report.md",
    val_perplexity=val_ppl,
    test_perplexity=test_ppl,
    samples=samples,
    step=step,
)
print(f"Report at: artifacts/reports/evaluation_report.md")

---
## Section 9 — Edge Export

Exports the trained model to:
1. HuggingFace format (`artifacts/models/hindi_slm_v001/`)
2. GGUF F16
3. GGUF Q4_K_M (requires `llama-quantize` in PATH)

Then validates Raspberry Pi 8 GB compatibility.

In [ ]:
from slm_training.exporter import (
    export_to_hf, convert_to_gguf_f16, quantize_gguf_q4km,
    validate_raspberry_pi_compatibility, write_export_report
)

HF_MODEL_DIR = MODELS_DIR / "hindi_slm_v001"
GGUF_DIR     = MODELS_DIR / "gguf"

# Step 1: HuggingFace format
model = model.cpu()  # move to CPU for saving
export_to_hf(model, tokenizer, MODEL_CFG, HF_MODEL_DIR)

In [ ]:
# Step 2: GGUF F16
# Requires llama.cpp convert script. See ACTION_ITEMS.md for install instructions.
gguf_f16_path = convert_to_gguf_f16(HF_MODEL_DIR, GGUF_DIR)

In [ ]:
# Step 3: Q4_K_M quantization
# Requires llama-quantize binary in PATH. See ACTION_ITEMS.md.
gguf_q4_path = quantize_gguf_q4km(gguf_f16_path, GGUF_DIR)

In [ ]:
# Step 4: Raspberry Pi compatibility validation
pi_report = validate_raspberry_pi_compatibility(
    hf_model_dir=HF_MODEL_DIR,
    model_cfg=MODEL_CFG,
    gguf_q4_path=gguf_q4_path,
)

# Write export report
write_export_report(
    report_path=REPORTS_DIR / "export_report.md",
    hf_dir=HF_MODEL_DIR,
    gguf_f16=gguf_f16_path,
    gguf_q4=gguf_q4_path,
    pi_report=pi_report,
)

print("\n=== Export Complete ===")
print(f"HF model : {HF_MODEL_DIR}")
print(f"GGUF F16 : {gguf_f16_path}")
print(f"GGUF Q4  : {gguf_q4_path}")
print(f"Pi fits  : {pi_report['fits_raspberry_pi_8gb']}")

---
## Training Complete!

### Summary of Artifacts

| Artifact | Path |
|----------|------|
| System profile | `configs/system_profile.yaml` |
| Model schema | `configs/optimal_model_schema.yaml` |
| Data plan | `configs/data_plan.yaml` |
| Tokenizer config | `configs/tokenizer_config.yaml` |
| Dataset manifest | `configs/tokenized_dataset_manifest.json` |
| Training config | `configs/training_config.yaml` |
| Checkpoints | `artifacts/checkpoints/step_*/` |
| HF model | `artifacts/models/hindi_slm_v001/` |
| GGUF F16 | `artifacts/models/gguf/hindi_slm_v001_f16.gguf` |
| GGUF Q4_K_M | `artifacts/models/gguf/hindi_slm_v001_q4_k_m.gguf` |
| Evaluation report | `artifacts/reports/evaluation_report.md` |
| Export report | `artifacts/reports/export_report.md` |
| TensorBoard logs | `logs/tensorboard/` |

### Next Steps
- Upload model to HuggingFace Hub: `huggingface-cli upload <your-hf-username>/hindi-slm-v001 artifacts/models/hindi_slm_v001/`
- Copy Q4_K_M GGUF to Raspberry Pi and test with `llama.cpp` server
- See `ACTION_ITEMS.md` for post-training tasks